# LLM + MCP dataset discovery

Natural-language prompts drive **real** `geopack_sdk_*` MCP tools (same handlers as Cursor).

**Jupyter:** uses **in-process** MCP (no stdio subprocess — avoids Windows `fileno` errors). Terminal/CLI uses stdio. Set `GEOPACK_MCP_MODE=stdio` to force subprocess.

**Patterns:**
- **§1** Deterministic geocode → list (no LLM)
- **§2** Two-stage: LLM extracts intent → MCP geocode → MCP list
- **§3** Single OpenAI tool loop calling MCP tools from `list_tools()`

**Prerequisites:** Geoportal API running, `notebooks/.env` with `GEOPACK_*` and `OPENAI_*`.

In [1]:
%pip install -q python-dotenv openai nest_asyncio
%pip install -q -e "../.[mcp,llm]"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

NOTEBOOK_DIR = Path.cwd()
SDK_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / "lib").is_dir() else NOTEBOOK_DIR
sys.path.insert(0, str(SDK_ROOT / "src"))
sys.path.insert(0, str(SDK_ROOT / "notebooks" / "lib"))

load_dotenv(SDK_ROOT / "notebooks" / ".env")
load_dotenv(SDK_ROOT / ".env")

from mcp_llm import (
    create_openai_client,
    deterministic_geocode_and_list,
    mcp_session,
    mcp_transport_mode,
    run_mcp_tool_loop,
    staged_dataset_search,
)

USER_PROMPT = "Find raster datasets in the Tehran area."
print("MCP transport:", mcp_transport_mode())
print("Prompt:", USER_PROMPT)

MCP transport: inprocess
Prompt: Find raster datasets in the Tehran area.


## 1. Deterministic MCP (no LLM)

Same chain as `test_mcp_stdio_client.py`: geocode Tehran → list rasters with bbox.

In [3]:
async def run_deterministic():
    async with mcp_session() as session:
        return await deterministic_geocode_and_list(
            session,
            place_query="Tehran, Iran",
            data_type="raster",
            start_date="2024-01-01",
            page_size=5,
        )

result_1 = await run_deterministic()
print("bbox:", result_1["geocode"].get("bbox"))
datasets = (result_1.get("list") or {}).get("datasets", [])
for row in datasets[:5]:
    print(f"- {row.get('name')} [id={row.get('id')}]")

bbox: [51.0892219, 35.5682071, 51.6063007, 35.8284702]
- Geology_1000K_PolygonToRaste_age.tif [id=1767]


## 2. Pattern A — two-stage (LLM intent → MCP)

Example: *Find raster datasets in the Tehran area.*

In [4]:
client = create_openai_client()

async def run_staged():
    async with mcp_session() as session:
        return await staged_dataset_search(
            session, client, USER_PROMPT, verbose=True
        )

result_2 = await run_staged()
print("Intent:", result_2["intent"])
datasets = (result_2.get("list") or {}).get("datasets", [])
print(f"Found {len(datasets)} dataset(s)")
for row in datasets[:10]:
    print(f"- {row.get('name')} [id={row.get('id')}] dataType={row.get('dataType')}")

[stage A] intent JSON: {
  "place_query": "Tehran, Iran",
  "data_type": "raster",
  "start_date": null,
  "end_date": null,
  "search_query": null,
  "page_size": 20
}
[stage B] geocode: {'display_name': 'شهر تهران, بخش مرکزی شهرستان تهران, شهرستان تهران, استان تهران, ایران', 'lat': 35.6892523, 'lon': 51.3896004, 'bbox': [51.0892219, 35.5682071, 51.6063007, 35.8284702]}
[stage C] list_datasets done
Intent: {'place_query': 'Tehran, Iran', 'data_type': 'raster', 'start_date': None, 'end_date': None, 'search_query': None, 'page_size': 20}
Found 1 dataset(s)
- Geology_1000K_PolygonToRaste_age.tif [id=1767] dataType=raster


## 3. Pattern B — single OpenAI + MCP tool loop (like Cursor)

Tool schemas come from the MCP server (`list_tools`), not hardcoded in the SDK.

In [5]:
async def run_loop():
    async with mcp_session() as session:
        return await run_mcp_tool_loop(
            session, client, USER_PROMPT, verbose=True
        )

result_3 = await run_loop()
print(result_3.get("assistant_text"))
print("\nTool calls:")
for step in result_3.get("tool_trace", []):
    print(" -", step["tool"])

[loop] 11 MCP tools: ['geopack_sdk_geocode_place', 'geopack_sdk_list_datasets', 'geopack_sdk_get_dataset', 'geopack_sdk_query_dataset', 'geopack_sdk_get_dataset_thumbnail', 'geopack_sdk_export_dataset', 'geopack_sdk_get_task', 'geopack_sdk_wait_for_task', 'geopack_sdk_list_workflows', 'geopack_sdk_get_workflow_run', 'geopack_sdk_download_generated_file']


HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 200 OK"


[loop round 1] geopack_sdk_geocode_place({'query': 'Tehran'})


HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 200 OK"


[loop round 2] geopack_sdk_list_datasets({'bbox': [51.0892219, 35.5682071, 51.6063007, 35.8284702], 'data_type': 'raster', 'details_level': 'lite'})


HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 200 OK"


[loop] final answer (3 rounds)
There is 1 raster dataset available in the Tehran area:

- Name: Geology_1000K_PolygonToRaste_age.tif
- Description: Geological raster dataset (thematic, single band) covering a wide area including Tehran, with age classifications for geological units.

If you need more details or want to download a preview, let me know!

Tool calls:
 - geopack_sdk_geocode_place
 - geopack_sdk_list_datasets


## 4. Optional — thumbnail via MCP

Uncomment and set `DATASET_ID` from the list above.

In [ ]:
# DATASET_ID = 2360
# async def fetch_thumb():
#     async with mcp_session() as session:
#         return await call_mcp_tool(
#             session,
#             "geopack_sdk_get_dataset_thumbnail",
#             {"dataset_id": DATASET_ID},
#         )
# thumb = await fetch_thumb()
# print(thumb)